# Historical preprocessing excerpt
제공된 CoolingPath_shadow.ipynb의 첫 9개 셀(전처리 부분)을 보존한 발췌본입니다. 실행 출력과 Colab 메타데이터는 제거했습니다. 원본 절대경로는 의도적으로 그대로 남겼으며 실행용 파이프라인은 scripts/preprocessing/와 docs/qgis-workflow.md를 사용하세요. 여러 건물의 중첩 면적을 합산하는 원본 방식에는 중복 집계 한계가 있습니다.

In [ ]:
pip install pysolar pytz pandas

In [ ]:
import pandas as pd
import pytz
from datetime import datetime
from pysolar.solar import get_altitude, get_azimuth

# ==========================================
# 1. 설정값 정의 (Configuration)
# ==========================================
# 날짜: 2025년 7월 8일 (화요일)
TARGET_DATE = "2025-07-08"

# 시간대: 08시 ~ 19시 (Total 12 hours)
START_HOUR = 8
END_HOUR = 19

# 위치: 서울 강남구 (Gangnam-gu, Seoul)
# 구글지도 등에서 확인 가능
LATITUDE = 37.5026   # 위도 (North)
LONGITUDE = 127.0427 # 경도 (East)

# 시간대 설정 (한국 표준시)
KST = pytz.timezone('Asia/Seoul')

# ==========================================
# 2. 태양 위치 계산 로직 (Calculator)
# ==========================================
solar_data = []

print(f"☀️ 태양 위치 시뮬레이션 시작")
print(f"- 날짜: {TARGET_DATE}")
print(f"- 위치: 위도 {LATITUDE}, 경도 {LONGITUDE}\n")

for hour in range(START_HOUR, END_HOUR + 1):
    # (1) 날짜와 시간을 결합하여 datetime 객체 생성 (KST 기준)
    # 예: 2025-07-08 08:00:00+09:00
    date_str = f"{TARGET_DATE} {hour}:00:00"
    dt_kst = KST.localize(datetime.strptime(date_str, "%Y-%m-%d %H:%M:%S"))

    # (2) Pysolar 라이브러리로 고도/방위각 계산
    # 주의: Pysolar는 UTC 시간을 기준으로 계산하므로, KST 객체를 그대로 넘기면 내부적으로 UTC로 변환해서 계산해줍니다.
    altitude = get_altitude(LATITUDE, LONGITUDE, dt_kst)
    azimuth = get_azimuth(LATITUDE, LONGITUDE, dt_kst)

    # (3) 결과 저장 (설계한 ERD 스키마인 정수형 time_slot 사용)
    solar_data.append({
        "time_slot": hour,          # PK: 8, 9, ... 19
        "sun_altitude": round(altitude, 2), # 소수점 2자리 반올림
        "sun_azimuth": round(azimuth, 2),
        "timestamp_kst": dt_kst     # (참고용) 실제 시간
    })

    print(f"[{hour:02d}:00] 고도: {altitude:.2f}°, 방위각: {azimuth:.2f}°")

# ==========================================
# 3. 결과 저장 (To CSV)
# ==========================================
df_solar = pd.DataFrame(solar_data)

# 보기 좋게 컬럼 순서 정렬
df_solar = df_solar[['time_slot', 'sun_altitude', 'sun_azimuth', 'timestamp_kst']]

# CSV 파일로 저장 (Step 2에서 불러와서 쓸 예정)
output_filename = "/content/drive/MyDrive/Colab Notebooks/data/solar_scenario_20250708.csv"
df_solar.to_csv(output_filename, index=False)

print(f"\n✅ 계산 완료! '{output_filename}' 파일이 생성되었습니다.")
print("-" * 50)
print(df_solar)

In [ ]:
pip install geopandas shapely numpy

In [ ]:
# [Cell 1] 기본 설정
import pandas as pd
import geopandas as gpd
import numpy as np
from shapely.geometry import Polygon, MultiPolygon, box
from shapely.ops import unary_union
import os

# 1. 파일 경로 설정 (Colab 경로 확인!)
SOLAR_CSV_PATH = "/content/drive/MyDrive/Colab Notebooks/data/solar_scenario_20250708.csv"
BUILDING_FILE_PATH = "/content/drive/MyDrive/Colab Notebooks/data/processed_buildings.shp"
OUTPUT_FILE_PATH = "/content/drive/MyDrive/Colab Notebooks/data/hourly_shadows_20250708.gpkg"

# 2. 컬럼 및 좌표계 설정
HEIGHT_COL = "height"   # 높이 컬럼명 확인 필수
TARGET_EPSG = 5179      # 미터 단위 좌표계 (UTMK)

print("✅ 라이브러리 및 설정 완료.")

In [ ]:
# [Cell 2 수정판] 잔상 기법(Iterative Translation)이 적용된 함수
def make_shadow_ghost(geometry, height, sun_alt, sun_azi, steps=5):
    """
    건물 형상, 높이, 태양 위치를 받아 '잔상 기법'으로 정밀한 그림자를 생성하는 함수
    :param steps: 잔상(Layer)의 개수. 높을수록 정밀하지만 느려짐. (기본값 5~10 추천)
    """
    # 해가 졌거나(0도 이하) 건물이 없으면 패스
    if sun_alt <= 0 or geometry is None: return None

    # 1. 그림자 총 길이 계산 (L = h / tan(고도))
    alt_rad = np.radians(max(sun_alt, 0.1))
    shadow_len = height / np.tan(alt_rad)

    # 2. 총 이동 거리 (dx, dy)
    azi_rad = np.radians(sun_azi)
    total_dx = -shadow_len * np.sin(azi_rad)
    total_dy = -shadow_len * np.cos(azi_rad)

    # ==========================================================
    # 🌟 [핵심] 잔상 기법 (Ghosting / Afterimage)
    # ==========================================================
    # 그림자가 이동하는 경로를 steps 만큼 쪼개서 중간 단계 건물을 만듭니다.
    # 예: steps=5 이면, 0%(바닥), 20%, 40%, 60%, 80%, 100%(지붕) 위치에 건물을 찍어냅니다.

    ghost_polys = []

    # MultiPolygon 처리
    parts = list(geometry.geoms) if isinstance(geometry, MultiPolygon) else [geometry]

    for part in parts:
        step_polys = []
        for i in range(steps + 1):
            ratio = i / steps  # 0.0 ~ 1.0

            # 비율만큼 조금씩 이동
            step_dx = total_dx * ratio
            step_dy = total_dy * ratio

            shifted = gpd.GeoSeries([part]).translate(xoff=step_dx, yoff=step_dy).iloc[0]
            step_polys.append(shifted)

        # 한 건물의 잔상들을 모두 합침 (Union)
        # Convex Hull을 쓰지 않고 Union을 쓰므로 오목한 모양이 유지됨
        merged_part = unary_union(step_polys)
        ghost_polys.append(merged_part)

    # 모든 파트 통합 반환
    return unary_union(ghost_polys)

print("✅ 잔상 기법(Ghosting) 함수 정의 완료.")

In [ ]:
# [Cell 3 수정판] 자르기 없이 로드 및 단순화만 수행
print("📂 데이터 로딩 중... (이미 잘라둔 파일 사용)")

# 1. 파일 읽기
try:
    df_solar = pd.read_csv(SOLAR_CSV_PATH)
    gdf_build = gpd.read_file(BUILDING_FILE_PATH)
except Exception as e:
    print(f"❌ 파일 로드 실패! 경로를 확인하세요: {e}")
    raise

# 2. 좌표계 변환 (필수)
if gdf_build.crs and gdf_build.crs.to_string() != f"EPSG:{TARGET_EPSG}":
    print(f"   - 좌표계 변환 중... ({gdf_build.crs} -> EPSG:{TARGET_EPSG})")
    gdf_build = gdf_build.to_crs(epsg=TARGET_EPSG)

# 3. 데이터 개수 확인 (검증용)
original_count = len(gdf_build)
print(f"🏢 로드된 건물 개수: {original_count}개")

if original_count > 3000:
    print("⚠️ [주의] 건물 개수가 3,000개를 넘습니다. 이미 잘라둔 파일이 맞는지 확인해주세요!")
    print("   (개수가 많으면 시뮬레이션 시간이 5분 이상 소요될 수 있습니다.)")

# 4. 형상 단순화 (Simplify) - 속도 향상 핵심 ⭐
# 건물의 모양을 0.3m 단위로 단순화합니다.
# 시각적으로는 차이가 없지만, 연산 속도는 2~3배 빨라집니다.
gdf_build['geometry'] = gdf_build.geometry.simplify(0.3)

print("-" * 30)
print(f"✅ 데이터 준비 완료! (단순화 적용됨)")
print("-" * 30)

In [ ]:
# [Cell 4 최종 수정판] DateTime 타입 적용
from tqdm.notebook import tqdm
import pandas as pd

print("🚀 그림자 생성 및 DateTime 변환 시작...")
all_shadows = []

# 날짜 기준일
BASE_DATE = "2025-07-08"

for idx, row in tqdm(df_solar.iterrows(), total=len(df_solar), desc="시간대별 처리"):
    time_slot = int(row['time_slot'])
    alt = row['sun_altitude']
    azi = row['sun_azimuth']

    # 1. 잔상 기법 적용 (steps=7)
    current_shadows = gdf_build.copy()
    current_shadows['geometry'] = current_shadows.apply(
        lambda x: make_shadow_ghost(x['geometry'], x[HEIGHT_COL], alt, azi, steps=7), axis=1
    )

    # 2. 메타데이터 추가
    current_shadows['time_slot'] = time_slot

    # 3. 날짜 문자열 생성 (예: "2025-07-08 14:00:00")
    time_str = f"{time_slot:02d}:00:00"
    current_shadows['timestamp'] = f"{BASE_DATE} {time_str}"

    # 빈 데이터 제거 및 수집
    current_shadows = current_shadows.dropna(subset=['geometry'])
    if not current_shadows.empty:
        all_shadows.append(current_shadows)

# 결과 저장
if all_shadows:
    gdf_final = pd.concat(all_shadows, ignore_index=True)

    # ⭐ [핵심 변경] 문자열(String) -> 진짜 날짜(DateTime) 객체로 변환
    print("⏳ 데이터 타입을 DateTime으로 변환 중...")
    gdf_final['timestamp'] = pd.to_datetime(gdf_final['timestamp'])

    # 저장
    gdf_final.to_file(OUTPUT_FILE_PATH, driver="GPKG", layer="hourly_shadows")

    print(f"\n🎉 저장 완료! 경로: {OUTPUT_FILE_PATH}")
    print("👉 QGIS에서 파일을 다시 열고 'timestamp' 필드 타입을 확인해보세요! (DateTime이어야 함)")
else:
    print("⚠️ 생성된 데이터가 없습니다.")

In [ ]:
# [Cell 5] 결과 검증 (시각화)
import matplotlib.pyplot as plt

# 1. 방금 만든 그림자 파일 다시 로드
print("🔍 결과 파일 검증 중...")
gdf_shadow_check = gpd.read_file(OUTPUT_FILE_PATH, layer="hourly_shadows")

# 2. CRS 확인 (아까 경고 떴으니 확인해봄)
print(f"🌍 좌표계 확인: {gdf_shadow_check.crs}")
if gdf_shadow_check.crs is None:
    print("   -> ⚠️ 좌표계가 유실되었습니다. EPSG:5179로 강제 설정합니다.")
    gdf_shadow_check.set_crs(epsg=TARGET_EPSG, inplace=True)

# 3. 14시(오후 2시) 데이터만 필터링
target_time = 14
shadow_14 = gdf_shadow_check[gdf_shadow_check['time_slot'] == target_time]
base_buildings = gpd.read_file(BUILDING_FILE_PATH) # 원본 건물

# 4. 시각화 (Matplotlib)
fig, ax = plt.subplots(figsize=(10, 10))

# 그림자 (회색, 반투명)
shadow_14.plot(ax=ax, color='gray', alpha=0.5, label='Shadow (14:00)')
# 건물 (검은색)
base_buildings.plot(ax=ax, color='black', alpha=1.0, label='Building')

plt.title(f"Shadow Simulation Check (Time: {target_time}:00)", fontsize=15)
plt.axis('off') # 축 제거
plt.legend()
plt.show()

print(f"✅ 14시 그림자 개수: {len(shadow_14)}개")

In [ ]:
#Step 3

import geopandas as gpd
import pandas as pd
from tqdm.notebook import tqdm

# ==========================================
# 1. 설정값 정의
# ==========================================
SHADOW_FILE = "/content/drive/MyDrive/Colab Notebooks/data/hourly_shadows_20250708.gpkg"  # Step 2 결과물
ROAD_FILE = "/content/drive/MyDrive/Colab Notebooks/data/processed_roads.shp"           # 도로 버퍼 파일 경로
OUTPUT_STAT_FILE = "/content/drive/MyDrive/Colab Notebooks/data/hourly_link_stat_20250708.csv" # 최종 통계 파일 (가벼운 CSV)

# 좌표계 설정 (미터 단위 필수)
TARGET_EPSG = 5179

# ==========================================
# 2. 데이터 로드
# ==========================================
print("📂 데이터 로딩 중...")

# (1) 도로 데이터 로드
gdf_road = gpd.read_file(ROAD_FILE)

# 🚨 좌표계(CRS)가 없는 경우 강제 설정
if gdf_road.crs is None:
    print(f"⚠️ 주의: 도로 데이터에 좌표계 정보(.prj)가 없습니다.")
    print(f"   -> EPSG:{TARGET_EPSG} (미터 단위)라고 가정하고 강제 설정합니다.")
    gdf_road.set_crs(epsg=TARGET_EPSG, inplace=True)

# 좌표계 변환 (혹시 다른 좌표계라면 변환)
elif gdf_road.crs.to_string() != f"EPSG:{TARGET_EPSG}":
    print(f"🔄 좌표계 변환: {gdf_road.crs} -> EPSG:{TARGET_EPSG}")
    gdf_road = gdf_road.to_crs(epsg=TARGET_EPSG)

# 도로 ID와 전체 면적 미리 계산 (분모로 사용)
# link_id 컬럼명이 맞는지 확인하세요! (예: LINK_ID, id, link_id 등)
LINK_ID_COL = "link_id"
gdf_road['total_area'] = gdf_road.geometry.area
road_base = gdf_road[[LINK_ID_COL, 'total_area', 'geometry']]

print(f"🛣 도로 개수: {len(gdf_road)}개")

# (2) 그림자 데이터 로드
gdf_shadow_all = gpd.read_file(SHADOW_FILE, layer="hourly_shadows")

# 🚨 좌표계(CRS)가 없는 경우 강제 설정
if gdf_shadow_all.crs is None:
    print(f"⚠️ 주의: 도로 데이터에 좌표계 정보(.prj)가 없습니다.")
    print(f"   -> EPSG:{TARGET_EPSG} (미터 단위)라고 가정하고 강제 설정합니다.")
    gdf_shadow_all.set_crs(epsg=TARGET_EPSG, inplace=True)

# 좌표계 변환 (혹시 다른 좌표계라면 변환)
elif gdf_shadow_all.crs.to_string() != f"EPSG:{TARGET_EPSG}":
    print(f"🔄 좌표계 변환: {gdf_shadow_all.crs} -> EPSG:{TARGET_EPSG}")
    gdf_shadow_all = gdf_shadow_all.to_crs(epsg=TARGET_EPSG)

# ==========================================
# 3. 공간 연산 (Overlay) 및 비율 계산
# ==========================================
print("🚀 시간대별 그늘 비율 계산 시작...")
final_stats = []

# 시간대별로 반복 (메모리 터짐 방지)
time_slots = sorted(gdf_shadow_all['time_slot'].unique())

for t in tqdm(time_slots, desc="Processing Hours"):
    # 해당 시간대 그림자만 필터링
    shadow_at_t = gdf_shadow_all[gdf_shadow_all['time_slot'] == t]

    # [핵심 로직] 공간 교차 (Intersection)
    # 도로(Road) 위에 그림자(Shadow)를 덮어서 겹치는 부분만 잘라냄
    # keep_geom_type=False : 겹쳐서 점이나 선이 되는 경우는 무시하고 '면'만 남김
    intersection = gpd.overlay(road_base, shadow_at_t, how='intersection', keep_geom_type=False)

    # 교차된 조각들의 면적 계산
    intersection['inter_area'] = intersection.geometry.area

    # [중요] 하나의 도로에 여러 그림자 조각이 있을 수 있으므로 ID별로 합침 (GroupBy)
    shadow_area_by_link = intersection.groupby(LINK_ID_COL)['inter_area'].sum().reset_index()
    shadow_area_by_link.rename(columns={'inter_area': 'shadow_area'}, inplace=True)

    # 도로 원본 정보와 합쳐서 비율 계산
    # (그림자가 하나도 없는 도로는 여기서 누락되므로, 나중에 0으로 채워야 함)
    merged = pd.merge(road_base[[LINK_ID_COL, 'total_area']], shadow_area_by_link, on=LINK_ID_COL, how='left')

    # 결측치(그림자 없음)는 0으로 채움
    merged['shadow_area'] = merged['shadow_area'].fillna(0)

    # 비율 계산 (0.0 ~ 1.0)
    merged['shadow_ratio'] = merged['shadow_area'] / merged['total_area']

    # 값이 1.0(100%)을 살짝 넘는 경우(부동소수점 오차) 1.0으로 보정
    merged.loc[merged['shadow_ratio'] > 1.0, 'shadow_ratio'] = 1.0

    # 메타데이터 추가
    merged['time_slot'] = t

    # 결과 수집 (ID, 시간, 비율) - 가볍게 숫자만 저장
    final_stats.append(merged[[LINK_ID_COL, 'time_slot', 'shadow_ratio']])

# ==========================================
# 4. 결과 저장
# ==========================================
df_final = pd.concat(final_stats, ignore_index=True)

# CSV로 저장 (DB에 넣기 좋게 가벼운 형태)
df_final.to_csv(OUTPUT_STAT_FILE, index=False)

print(f"\n🎉 계산 완료! 결과 파일: {OUTPUT_STAT_FILE}")
print(f"📊 총 데이터 행 수: {len(df_final)}")
print("\n[미리보기]")
print(df_final.head())